In [1]:
import pandas as pd
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,LSTM,Embedding,Dropout,Input


In [2]:
df=pd.read_csv("SMSSpamCollection",sep="\t",names=['result','email'])

In [3]:
df

,result,email
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [4]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df['result']=le.fit_transform(df['result'])

In [5]:
import re
def text_clean(text):
    text=str.lower(text)
    text=re.sub("[^a-zA-Z0-9]"," ",text)
    return text
    


In [6]:
df['email']=df['email'].apply(text_clean)

In [7]:
df.head(6)

,result,email
0,0,go until jurong point crazy available only ...
1,0,ok lar joking wif u oni
2,1,free entry in 2 a wkly comp to win fa cup fina...
3,0,u dun say so early hor u c already then say
4,0,nah i don t think he goes to usf he lives aro...
5,1,freemsg hey there darling it s been 3 week s n...


In [8]:
from sklearn.model_selection import train_test_split
X=df['email']
y=df['result']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3)

In [9]:
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer=Tokenizer(num_words=10000)

In [10]:
tokenizer.fit_on_texts(X_train)

In [11]:
X_train_seq=tokenizer.texts_to_sequences(X_train)
X_test_seq=tokenizer.texts_to_sequences(X_test)

In [12]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

    
    

In [13]:
X_train_pad=pad_sequences(
    X_train_seq,
    maxlen=200,
    padding='post',
    truncating='post'
)
X_test_pad=pad_sequences(
    X_test_seq,
    maxlen=200,
    padding='post',
    truncating='post'
)

In [14]:
model=Sequential([
    Input(shape=(200,)),
    Embedding(10000,64),
    LSTM(128),
    Dense(128,activation='relu'),
    Dropout(0.2),
    Dense(1,activation='sigmoid')
    
          
])

In [15]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [16]:
model.fit(X_train_pad,y_train,epochs=5,validation_split=0.2,batch_size=32)

Epoch 1/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 26s 232ms/step - accuracy: 0.8548 - loss: 0.4269 - val_accuracy: 0.8731 - val_loss: 0.3877
Epoch 2/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 40s 217ms/step - accuracy: 0.8615 - loss: 0.4068 - val_accuracy: 0.8731 - val_loss: 0.3809
Epoch 3/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 21s 215ms/step - accuracy: 0.8615 - loss: 0.4070 - val_accuracy: 0.8731 - val_loss: 0.3820
Epoch 4/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 29s 299ms/step - accuracy: 0.8615 - loss: 0.4054 - val_accuracy: 0.8731 - val_loss: 0.3832
Epoch 5/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 30s 302ms/step - accuracy: 0.8615 - loss: 0.4094 - val_accuracy: 0.8731 - val_loss: 0.3805


In [17]:
loss,accuaracy=model.evaluate(X_test_pad,y_test)

53/53 ━━━━━━━━━━━━━━━━━━━━ 4s 78ms/step - accuracy: 0.8708 - loss: 0.3848


In [18]:

pred=model.predict(X_test_pad[7:8])
if(pred>0.5):
    print("prediction: Spam")
else:
    print("prediction: not spam")
actual=y_test[7]
if (actual>=0.5):
    print("Actual: spam")
else:
    print("Actual: not spam")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 398ms/step
prediction: not spam
Actual: not spam
